# Experiment no 3

## Objective
Build a multiple linear regression model using the supplied `autos_dataset.csv` dataset to predict automobile prices. Assess predictive performance on a held-out test set using **MSE**, **RMSE**, and **R2**.

## Workflow
1. Load and inspect the dataset.
2. Clean missing-value markers (`?`) and convert numeric-looking columns to numeric types.
3. Separate the target (`price`) from input features.
4. Split the data into training and testing sets.
5. Preprocess numeric and categorical features using imputation and one-hot encoding.
6. Fit a multiple linear regression model.
7. Evaluate predictions and inspect actual vs. predicted prices.


### Step 1: Import Libraries and Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

RANDOM_STATE = 42
print("Dependencies loaded successfully.")


### Step 2: Load Data


In [ ]:
import os
import urllib.request
import ssl

csv_path = "autos_dataset.csv"
if not os.path.exists(csv_path):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
    columns = [
        "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
        "num-of-doors", "body-style", "drive-wheels", "engine-location",
        "wheel-base", "length", "width", "height", "curb-weight",
        "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
        "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
        "city-mpg", "highway-mpg", "price"
    ]
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.urlopen(url, context=ctx)
    df_raw = pd.read_csv(req, header=None, names=columns)
    df_raw.to_csv(csv_path, index=False)

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


### Step 3: Data Inspection and Cleaning


In [ ]:
# The source dataset uses '?' to mark some missing values.
df = df.replace("?", np.nan)

# Convert object columns to numeric when their non-missing values are numeric.
# This handles columns such as horsepower, normalized-losses, and price.
for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        original_nonmissing = df[col].notna().sum()
        converted_nonmissing = converted.notna().sum()
        if original_nonmissing > 0 and converted_nonmissing / original_nonmissing > 0.5:
            df[col] = converted

# Price is the prediction target; rows without a known price cannot train/evaluate
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"]).copy()

print("Shape after cleaning target:", df.shape)
print("\nMissing values in columns:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(10))
display(df.describe(include="all").T)


### Step 4: Exploratory Data Analysis


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["price"], kde=True)
plt.title("Distribution of Automobile Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Explore relationships between a few commonly useful numeric predictors and price.
candidate_cols = [c for c in ["engine-size", "horsepower", "curb-weight", "highway-mpg"] if c in df.columns]
for col in candidate_cols:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=df, x=col, y="price")
    plt.title(f"Price vs. {col}")
    plt.tight_layout()
    plt.show()


### Step 5: Prepare Features and Split the Data


In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])


### Step 6: Train the Multiple Linear Regression Model


In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Model training completed.")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")


### Step 7: Evaluate Predictive Capability


In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "R2", "MAE"],
    "Value": [mse, rmse, r2, mae],
    "Interpretation": [
        "Average squared prediction error (price units squared); lower is better.",
        "Typical error scale in the same units as price; lower is better.",
        "Proportion of test-target variance explained; closer to 1 is generally better.",
        "Average absolute prediction error in price units; lower is better."
    ]
})
display(metrics)


### Step 8: Compare Actual and Predicted Prices


In [ ]:
comparison = pd.DataFrame({
    "Actual price": y_test.reset_index(drop=True),
    "Predicted price": pd.Series(y_pred)
})
comparison["Residual (actual - predicted)"] = (
    comparison["Actual price"] - comparison["Predicted price"]
)
display(comparison.head(15).round(2))


### Step 9: Save Test Predictions (Optional)


In [ ]:
comparison.to_csv("autos_price_predictions.csv", index=False)
print("Saved predictions to autos_price_predictions.csv")


## Interpretation and Conclusion
- **MSE** penalizes larger errors more strongly because residuals are squared.
- **RMSE** is in the same units as the dataset's price column, making it easy to interpret.
- **R2** compares the model against predicting the mean price. A value close to 1 indicates strong explanatory performance on the test split.
- **MAE** provides an additional directly interpretable average absolute error.

Linear regression provides an interpretable baseline; however, multi-collinear or highly skewed categorical features can benefit from regularized models or non-linear regressors.
